Project: Delivery Order Data Preparation
Goal

A delivery company has:

Drivers
Customer orders

We use pandas to answer:

Which orders are possible to deliver and which drivers are available?

In [40]:
import pandas as pd
orders=pd.read_csv("orders.csv")
drivers=pd.read_csv("drivers.csv")
print(orders)
print(drivers)

  order_id customer_zone  distance_km  order_value     status
0       O1         North            3           20    pending
1       O2         South           10           35    pending
2       O3         North            5           15  cancelled
3       O4       Central           20           40    pending
4       O5         South            8           25    pending
5       O6         North            2           18    pending
6       O7       Central           12           30    pending
7       O8         South            4           22    pending
8       O9         North            7           19    pending
9      O10       Central            9           27    pending
  driver_id   name     zone  max_orders  available
0        D1    Ali    North           4          1
1        D2   Sara    South           3          1
2        D3   John  Central           5          1
3        D4   Mina    North           4          0
4        D5  Ahmed    South           5          1
5        D6 

In [41]:
orders=orders[orders["status"]=="pending"]
print(orders)

  order_id customer_zone  distance_km  order_value   status
0       O1         North            3           20  pending
1       O2         South           10           35  pending
3       O4       Central           20           40  pending
4       O5         South            8           25  pending
5       O6         North            2           18  pending
6       O7       Central           12           30  pending
7       O8         South            4           22  pending
8       O9         North            7           19  pending
9      O10       Central            9           27  pending


In [42]:
orders=orders[orders["distance_km"]<=12]
print(orders)
print("Remaining order",len(orders))

  order_id customer_zone  distance_km  order_value   status
0       O1         North            3           20  pending
1       O2         South           10           35  pending
4       O5         South            8           25  pending
5       O6         North            2           18  pending
6       O7       Central           12           30  pending
7       O8         South            4           22  pending
8       O9         North            7           19  pending
9      O10       Central            9           27  pending
Remaining order 8


In [43]:
orders["dilivery cost"]=orders['distance_km']*2
print(orders)

  order_id customer_zone  distance_km  order_value   status  dilivery cost
0       O1         North            3           20  pending              6
1       O2         South           10           35  pending             20
4       O5         South            8           25  pending             16
5       O6         North            2           18  pending              4
6       O7       Central           12           30  pending             24
7       O8         South            4           22  pending              8
8       O9         North            7           19  pending             14
9      O10       Central            9           27  pending             18


In [44]:
orders=orders.sort_values(by="order_value",ascending=False)
print(orders)

  order_id customer_zone  distance_km  order_value   status  dilivery cost
1       O2         South           10           35  pending             20
6       O7       Central           12           30  pending             24
9      O10       Central            9           27  pending             18
4       O5         South            8           25  pending             16
7       O8         South            4           22  pending              8
0       O1         North            3           20  pending              6
8       O9         North            7           19  pending             14
5       O6         North            2           18  pending              4


In [46]:
group = orders.groupby("customer_zone")
print(group["order_id"].count())

customer_zone
Central    2
North      3
South      3
Name: order_id, dtype: int64


# Food Delivery Scheduling Project – Pandas Data Cleaning and Filtering

## Objective

Before building a scheduling model with OR-Tools, the raw delivery order data must be cleaned, filtered, and analyzed. Pandas is used to prepare the data so that only valid and useful orders are sent to the optimization model.

---

## Step 1: Read the CSV File

The first step is to load the dataset into a Pandas DataFrame.

```python
import pandas as pd

orders = pd.read_csv("orders.csv")
```

This converts the CSV file into a structured table that can be manipulated and analyzed.

---

## Step 2: Inspect the Dataset

Before performing any analysis, it is important to understand the structure of the dataset.

```python
print(orders.head())
print(orders.info())
print(orders.columns)
```

These commands help identify:

* Column names
* Data types
* Number of rows
* Missing values

---

## Step 3: Check for Missing Values

Real-world datasets often contain incomplete information.

```python
print(orders.isnull().sum())
```

If missing values exist, they can be handled using:

```python
orders = orders.dropna()
```

or

```python
orders["distance_km"] = orders["distance_km"].fillna(0)
```

Data cleaning is important because optimization models require complete and reliable data.

---

## Step 4: Remove Cancelled Orders

Only orders that still require delivery should be considered.

```python
orders = orders[orders["status"] == "pending"]
```

This filtering step removes cancelled orders from the dataset.

---

## Step 5: Filter Orders by Distance

Suppose the company only delivers within a 10 km radius.

```python
orders = orders[orders["distance_km"] <= 10]
```

This removes orders that are outside the service area.

---

## Step 6: Filter High-Value Orders

Management may want to prioritize more profitable deliveries.

```python
orders = orders[orders["order_value"] >= 20]
```

This keeps only orders with a value greater than or equal to 20.

---

## Step 7: Apply Multiple Business Rules

Several constraints can be applied simultaneously.

```python
orders = orders[
    (orders["status"] == "pending") &
    (orders["distance_km"] <= 10) &
    (orders["order_value"] >= 20)
]
```

This ensures that only valid, nearby, and profitable orders remain.

---

## Step 8: Create a New Cost Column

A delivery cost can be estimated based on distance.

```python
orders["delivery_cost"] = orders["distance_km"] * 2
```

This creates a new feature that may later be used in the optimization objective function.

---

## Step 9: Sort Orders by Priority

Orders can be sorted by value to identify the most important deliveries.

```python
orders = orders.sort_values(
    by="order_value",
    ascending=False
)
```

This displays the highest-value orders first.

---

## Step 10: Analyze Demand by Zone

Count the number of orders in each delivery zone.

```python
orders.groupby("customer_zone")["order_id"].count()
```

This helps identify areas with high demand.

---

## Step 11: Calculate Revenue by Zone

Determine how much revenue is generated in each zone.

```python
orders.groupby("customer_zone")["order_value"].sum()
```

This helps identify the most profitable delivery areas.

---

## Step 12: Calculate Average Delivery Distance

Analyze delivery difficulty in each zone.

```python
orders.groupby("customer_zone")["distance_km"].mean()
```

This provides insight into the average travel distance required.

---

## Step 13: Save the Cleaned Dataset

After cleaning and filtering, save the processed data.

```python
orders.to_csv("clean_orders.csv", index=False)
```

The resulting file can be used as input for scheduling and optimization models.

---

## Scheduling Workflow

Raw Orders Data

↓

Data Inspection

↓

Missing Value Handling

↓

Filtering and Cleaning

↓

Feature Engineering

↓

Demand Analysis

↓

Save Clean Dataset

↓

OR-Tools Scheduling Model

The purpose of Pandas in this project is to transform raw business data into a clean and structured dataset that can be used for optimization and decision-making.
